In [1]:
import os, sys, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
sys.path.insert(0, os.getcwd())
try:
    from 集成公共函数 import *
except ImportError as e:
    print('导入 集成公共函数.py 失败:', e)
    raise

In [2]:
# 区间预测参数
ALPHA = 0.10          #名义缺失率: 90%预测区间
SEED  = 0
print(f'区间参数: 名义置信水平 = {(1-ALPHA)*100:.0f}% | 随机种子 = {SEED}')

区间参数: 名义置信水平 = 90% | 随机种子 = 0


In [3]:
ART = load_or_build_models()         
ENSEMBLES     = ART['ENSEMBLES']
TUNED         = ART['TUNED']
W_raw         = ART['W_raw']
WEIGHTS_TABLE = ART['WEIGHTS_TABLE']
CV_SCORES     = ART['CV_SCORES']
BEST_PARAMS   = ART['BEST_PARAMS']
TUNE_META     = ART['TUNE_META']


print('已加载目标:', list(ENSEMBLES.keys()))
print('成员数:', len(base_learners()), '| 各目标非零权重成员数:',
      {(TAG_CN[k], int((W_raw[k] > 0.001).sum())) for k in ENSEMBLES})

[加载] 已从存档载入集成模型: d:\Code_File\Python_Data_AI\Project_CFPA_Teacher\Data\集成建模\集成模型存档.joblib (version=3)
已加载目标: ['TS', 'TM', 'FS', 'FM', 'IS']
成员数: 12 | 各目标非零权重成员数: {('弹性模量', 4), ('冲击强度', 3), ('弯曲模量', 5), ('弯曲强度', 3), ('拉伸强度', 4)}


In [4]:
# 嵌套OOF成员矩阵
NESTED = ART.get('NESTED', None)
if NESTED is None:
    print('存档中没有 NESTED 重算')
    NESTED = {}
    for tag in SHEETS.values():
        t0 = time.time()
        NESTED[tag] = nested_oof_v2(tag, tuned=TUNED[tag], nested_tune=NESTED_TUNE,
                                    n_outer=N_OUTER, n_inner=N_INNER, verbose=False)
        print(f"  [{tag}] {TAG_CN[tag]} 完成({time.time()-t0:.0f}s) "
              f"OOF-R2={r2_score(NESTED[tag]['y'], NESTED[tag]['oof']):.3f}")

        
else:
    print('[复用] 从模块一存档载入诚实嵌套OOF成员矩阵:',
          {t: (NESTED[t]['group_mode'], bool(NESTED[t]['nested_tune'])) for t in SHEETS.values()})
DIAG = NESTED
for tag in SHEETS.values():
    print(f"  [{tag}] {TAG_CN[tag]} OOF-R2={r2_score(DIAG[tag]['y'], DIAG[tag]['oof']):.3f}"
          f" | Pm shape={DIAG[tag]['Pm'].shape} | 成员数={len(DIAG[tag]['names'])}")

[复用] 从模块一存档载入诚实嵌套OOF成员矩阵: {'TS': ('combo', True), 'TM': ('combo', True), 'FS': ('combo', True), 'FM': ('combo', True), 'IS': ('combo', True)}
  [TS] 拉伸强度 OOF-R2=0.341 | Pm shape=(186, 12) | 成员数=12
  [TM] 弹性模量 OOF-R2=0.280 | Pm shape=(186, 12) | 成员数=12
  [FS] 弯曲强度 OOF-R2=0.779 | Pm shape=(186, 12) | 成员数=12
  [FM] 弯曲模量 OOF-R2=0.571 | Pm shape=(186, 12) | 成员数=12
  [IS] 冲击强度 OOF-R2=0.332 | Pm shape=(186, 12) | 成员数=12


In [5]:
#成员分位数区间函数
def member_predictions(ensemble, X):
    '''12个成员在X上的预测矩阵 (n, 12) —— 部署用全数据拟合后的成员'''
    X = np.asarray(X, float)
    return np.column_stack([m.predict(X) for m in ensemble.fitted_.values()])

def quantile_interval(Pm, alpha):
    '''对成员预测矩阵(n,L)取经验分位数 [α/2, 1-α/2] 作为预测区间'''
    lo = np.quantile(Pm, alpha/2.0, axis=1)
    hi = np.quantile(Pm, 1-alpha/2.0, axis=1)
    return lo, hi


#训练口径: 用嵌套OOF成员预测矩阵
TRAIN_IV = {}
for tag in SHEETS.values():
    lo, hi = quantile_interval(DIAG[tag]['Pm'], ALPHA)
    TRAIN_IV[tag] = (lo, hi, DIAG[tag]['oof'])

# 外部口径: 用全数据拟合成员
EXT_IV = {}
for tag in SHEETS.values():
    Xe = load_external()[tag][FEATS].values.astype(float)
    Pm = member_predictions(ENSEMBLES[tag], Xe)
    lo, hi = quantile_interval(Pm, ALPHA)
    EXT_IV[tag] = (lo, hi, ENSEMBLES[tag].predict(Xe))

In [6]:
# 指标汇总表
rows = []
for tag in SHEETS.values():
    y_tr = DIAG[tag]['y']
    ye   = load_external()[tag]['y_mean'].values.astype(float)
    lo, hi, mu = TRAIN_IV[tag]
    tr = interval_metrics(y_tr, lo, hi)
    lo, hi, mu = EXT_IV[tag]
    ex = interval_metrics(ye, lo, hi)
    rows.append(dict(目标=TAG_CN[tag],
                     训练_PICP=round(tr['PICP'],3), 训练_MPIW=round(tr['MPIW'],2), 训练_PINAW=round(tr['PINAW'],3),
                     外部_PICP=round(ex['PICP'],3), 外部_MPIW=round(ex['MPIW'],2), 外部_PINAW=round(ex['PINAW'],3)))
metric_table = pd.DataFrame(rows)
metric_table

,目标,训练_PICP,训练_MPIW,训练_PINAW,外部_PICP,外部_MPIW,外部_PINAW
0,拉伸强度,0.414,10.16,0.189,0.750,9.04,0.472
1,弹性模量,0.414,1517.42,0.141,0.375,1025.10,0.336
2,弯曲强度,0.613,19.05,0.193,0.625,11.17,0.265
3,弯曲模量,0.425,787.29,0.177,0.500,613.22,0.200
4,冲击强度,0.296,2.50,0.130,0.625,2.01,0.247


In [7]:
#测试 对新工艺参数输出点预测 + 成员分歧区间
x = np.array([[310, 90, 30, 0.1, 1]])      # 喷嘴温度310°C, 热床90°C, 速度30, 层厚0.1mm, 方向1
print('特征顺序:', FEATS)
for tag in SHEETS.values():
    Pm = member_predictions(ENSEMBLES[tag], x)
    lo, hi = quantile_interval(Pm, ALPHA)
    mu = ENSEMBLES[tag].predict(x)
    print(f"{TAG_CN[tag]:6s} ({tag}): 点预测={mu[0]:7.2f} | {(1-ALPHA)*100:.0f}%区间=[{lo[0]:7.2f}, {hi[0]:7.2f}] 宽={hi[0]-lo[0]:.2f}")

特征顺序: ['T_noz', 'T_bed', 'speed', 'layer', 'direction']
拉伸强度   (TS): 点预测= 130.14 | 90%区间=[ 123.21,  134.27] 宽=11.06
弹性模量   (TM): 点预测=14510.25 | 90%区间=[14187.12, 15205.62] 宽=1018.50
弯曲强度   (FS): 点预测= 165.65 | 90%区间=[ 164.59,  167.54] 宽=2.95
弯曲模量   (FM): 点预测=9097.75 | 90%区间=[8877.65, 9209.99] 宽=332.34
冲击强度   (IS): 点预测=  26.81 | 90%区间=[  25.40,   27.89] 宽=2.49
